# 9 · What do the numbers mean in micrometres and days?

All notebooks work in grid units. To compare with biology, two choices are needed: how large a grid cell is, and how long one time unit lasts. These are **assumptions**, set in the first code cell, and everything below follows from them:

- **One grid cell = 10 µm**, about the diameter of one tumour cell.
- **Cell doubling time = 24 h.** Jafari Nivlouei et al. (2021, [doi:10.1371/journal.pcbi.1009081](https://doi.org/10.1371/journal.pcbi.1009081)) base their time scale on a fastest cell-cycle time of about 24 h. With $\alpha_0 = 1$ per time unit, a sparse tumour doubles every $\ln 2$ time units, so one time unit is $24/\ln 2 \approx 34.6$ h.

For a reference length, Thomlinson & Gray (1955, *Br J Cancer* 9: 539, [doi:10.1038/bjc.1955.55](https://doi.org/10.1038/bjc.1955.55)) measured human lung tumour cords. They found that "no apparently intact tumour cell is seen more than 180 µ from the stroma", with critical radii averaging 169 µm, against 145 µm calculated from oxygen diffusion.

In [1]:
import sys
sys.path.insert(0, "..")    # shared code lives in corrected/angiogenesis.py
import numpy as np
import matplotlib.pyplot as plt
import angiogenesis as ag

In [2]:
CELL_UM = 10.0                          # assumption: one grid cell = 10 micrometres
DOUBLING_H = 24.0                       # assumption: tumour cell doubling time in hours
TIME_UNIT_H = DOUBLING_H / np.log(2)    # alpha0 = 1 per time unit -> doubling time = ln 2 units
DAY_PER_UNIT = TIME_UNIT_H / 24
print(f"1 grid cell = {CELL_UM:g} um,  1 time unit = {TIME_UNIT_H:.1f} h = {DAY_PER_UNIT:.2f} days")

1 grid cell = 10 um,  1 time unit = 34.6 h = 1.44 days


In [3]:
# name: (grid size l, D, alpha at the front, number of steps, delta_t)
models = {
    "1 diffusion, one region":        (100, 1.0,   0, 60,   0.1),
    "2 diffusion, three regions":     (200, 2.0,   0, 2000, 0.1),
    "3 proliferation, small D":       (200, 0.001, 2, 2000, 0.1),
    "4 diffusion + proliferation":    (200, 0.05,  1, 1000, 0.1),
    "5 irrigation":                   (200, 0.2,   1, 1000, 0.1),
    "6 nutrient-limited (extension)": (200, 0.2,   1, 1000, 0.1),
}
speed_um_per_day = CELL_UM / DAY_PER_UNIT          # 1 grid cell per time unit, in um/day

print(f"{'model':32s} {'domain':>8s} {'simulated':>10s} {'D':>10s} {'front speed (um/day)':>24s}")
print(f"{'':32s} {'(mm)':>8s} {'(days)':>10s} {'(um^2/h)':>10s} {'correct':>11s} {'on grid':>11s}")
for name, (l, D, alpha, steps, dt) in models.items():
    domain_mm = l * CELL_UM / 1000
    days = steps * dt * DAY_PER_UNIT
    D_phys = D * CELL_UM**2 / TIME_UNIT_H
    if alpha:
        correct = ag.continuum_speed(D, alpha) * speed_um_per_day
        on_grid = ag.lattice_speed(D, alpha, 1.0) * speed_um_per_day
        speeds = f"{correct:11.2f} {on_grid:11.2f}"
    else:
        speeds = f"{'-':>11s} {'-':>11s}"
    print(f"{name:32s} {domain_mm:8.1f} {days:10.1f} {D_phys:10.3g} {speeds}")

model                              domain  simulated          D     front speed (um/day)
                                     (mm)     (days)   (um^2/h)     correct     on grid
1 diffusion, one region               1.0        8.7       2.89           -           -
2 diffusion, three regions            2.0      288.5       5.78           -           -
3 proliferation, small D              2.0      288.5    0.00289        0.62        2.77
4 diffusion + proliferation           2.0      144.3      0.144        3.10        4.20
5 irrigation                          2.0      144.3      0.578        6.20        7.06
6 nutrient-limited (extension)        2.0      144.3      0.578        6.20        7.06


In [4]:
# Notebook 06: how far does nutrient reach into a dense tumour?
L_cells = 8                                   # penetration length used in notebook 06
print(f"notebook 06 nutrient penetration: {L_cells} cells = {L_cells * CELL_UM:.0f} um")
print("Thomlinson & Gray (1955): viable rim <= 180 um, critical radius ~169 um (145 um calculated)")
print(f"to match ~169 um at {CELL_UM:g} um per cell, L would be about {169 / CELL_UM:.0f} cells")

# Only L = sqrt(D_c / lambda) matters, because the nutrient is kept close to equilibrium.
D_c_grid = 40
D_c_phys = D_c_grid * CELL_UM**2 / TIME_UNIT_H
D_oxygen = 2.0e-5 * 1e8 * 3600                # 2e-5 cm^2/s (Thomlinson & Gray) in um^2/h
print(f"notebook 06 D_c = {D_c_phys:.0f} um^2/h vs oxygen {D_oxygen:.2e} um^2/h "
      f"({D_oxygen / D_c_phys:,.0f}x faster)")

notebook 06 nutrient penetration: 8 cells = 80 um
Thomlinson & Gray (1955): viable rim <= 180 um, critical radius ~169 um (145 um calculated)
to match ~169 um at 10 um per cell, L would be about 17 cells
notebook 06 D_c = 116 um^2/h vs oxygen 7.20e+06 um^2/h (62,324x faster)


**Reading the result.**

- **Scale.** With these assumptions every 2D grid is a patch of tissue 1–2 mm across. The growth models (3–6) cover about 140–290 days, and the corrected tumour fronts advance by roughly 0.6–6 µm per day.
- **Grid error in physical units.** The "on grid" column shows the actual speeds at the spacing used ([notebook 8](08-grid-refinement.ipynb)). In model 3 the grid makes the front about 4× too fast.
- **Nutrient reach (notebook 06).** Its penetration length of 8 cells is 80 µm, about half of what Thomlinson & Gray measured for oxygen in tumours. Raising $L$ to about 17 cells would make model 06's viable rim realistic at 10 µm per cell.
- **The nutrient diffusion constant itself does not need to be realistic.** Notebook 06 keeps the nutrient close to equilibrium, so only the ratio $\sqrt{D_c/\lambda}$ matters. That is why its $D_c$ can be tens of thousands of times smaller than oxygen's real diffusion constant without affecting the result.
- **Everything here scales with the two assumptions.** Change `CELL_UM` or `DOUBLING_H` above and re-run to see how the conversions move.